In [ ]:
import os
import tensorflow as tf
import numpy as np
from numpy.linalg import LinAlgError
from scipy.linalg import block_diag, qr
from matplotlib import pyplot as plt
import random
import math

In [ ]:
rng = np.random.default_rng(seed=42)

In [ ]:
def generate_eigvals(d, positive_halfplane=False):
    eigvals = []
    if d % 2 == 1:
        eigvals.append(np.random.uniform(-1, 1))
    for i in range(d // 2):
        r = np.random.uniform(-1, 1)
        phi = np.random.uniform(0, math.pi)
        eigvals.append(r * np.exp(1j * phi))
        eigvals.append(r * np.exp(-1j * phi))
    if positive_halfplane:
        eigvals = [x if x.real > 0 else -x for x in eigvals]
    return np.array(eigvals)

In [ ]:
p = 0.3


def generate_matrix(eig_list, randomize=True, seed=None):
    """
    Given a list of d complex numbers (eig_list), return a real d×d matrix A
    whose eigenvalues are exactly those numbers (with multiplicity).

    Conditions on eig_list:
      - len(eig_list) = d
      - Any non-real eigenvalue must be paired with its complex conjugate.

    If randomize=True, A = Q J Q^T for a random orthogonal Q, where J is the real
    block-diagonal form.  If randomize=False, A is simply the block-diagonal J itself.

    Returns:
      A_real : an ndarray of shape (d,d), dtype=float.
    """
    # 1) Verify length and conjugate‐closure:
    d = len(eig_list)
    eig_array = np.asarray(eig_list, dtype=complex)
    # Count occurrences of each value
    # We will mark which indices have been “used” in pairing.
    used = np.zeros(d, dtype=bool)

    # Build blocks (as Python lists) that will later become a block diagonal matrix.
    blocks = []

    for i in range(d):
        if used[i]:
            continue
        lam = eig_array[i]
        if np.isclose(lam.imag, 0.0):
            # Real eigenvalue → make a 1×1 block [λ.real]
            blocks.append(np.array([[lam.real]], dtype=float))
            used[i] = True
        else:
            # Non‐real: look for its exact conjugate (with tolerance)
            conj = lam.conjugate()
            # find an index j!=i s.t. eig_array[j] ≈ conj
            found = False
            for j in range(i + 1, d):
                if (not used[j]) and np.allclose(eig_array[j], conj):
                    # build 2×2 real block for {α ± iβ}:
                    α = lam.real
                    β = lam.imag
                    block = np.array([[α, -β], [β, α]], dtype=float)
                    blocks.append(block)
                    used[i] = True
                    used[j] = True
                    found = True
                    break
            if not found:
                raise ValueError(
                    f"Eigenvalue {lam:.4g} at index {i} is non‐real but its conjugate "
                    "is not in the list with matching multiplicity."
                )

    # Double-check we used exactly d entries
    if not np.all(used):
        raise RuntimeError("Internal error: some eigenvalues were not paired/used.")

    # 2) Form the block‐diagonal matrix J
    #    (block_diag(*blocks) stacks all given blocks along the diagonal)
    J = block_diag(*blocks)

    if not randomize:
        # Return J directly (still real, but block‐diagonal in 1×1 and 2×2 blocks).
        return J

    # 3) If randomize=True: conjugate by a random orthogonal Q
    if seed is not None:
        np.random.seed(seed)
    # Draw random normal matrix, do QR ⇒ Q is uniform orthogonal
    X = np.random.randn(d, d)

    for _ in range(100):
        Y = np.random.choice([0, 1], size=(d, d), p=[1 - p, p])
        if np.linalg.det(Y) > 1e-7:
            X = X @ Y
            X_inv = np.linalg.inv(X)
            break
        else:
            continue

    A = X @ J @ X_inv
    return A

In [ ]:
d = 16  # matrix dimension
dataset_size = 10000

X = np.ndarray(shape=(2 * dataset_size, d, d))

for i in range(dataset_size):
    eigvals = np.diag(generate_eigvals(d, False))
    Q, _ = np.linalg.qr(np.random.randn(d, d))
    A = Q @ eigvals @ Q.T
    X[i] = A

for i in range(dataset_size, 2 * dataset_size):
    eigvals = np.diag(generate_eigvals(d, True))
    Q, _ = np.linalg.qr(np.random.randn(d, d))
    A = Q @ eigvals @ Q.T
    X[i] = A

y = np.array([0] * dataset_size + [1] * dataset_size)

# 0 - eigenvalues in both positive and negative halfplane
# 1 - eigvals in positive halfplane only


# print(X)
# print(y)

<ipython-input-10-460410128>:10: ComplexWarning: Casting complex values to real discards the imaginary part
  X[i] = A
<ipython-input-10-460410128>:16: ComplexWarning: Casting complex values to real discards the imaginary part
  X[i] = A


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Flatten(input_shape=(d, d)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(2),
    ]
)

model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model.fit(X_train, y_train, epochs=10)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9410 - loss: 0.1894
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9844 - loss: 0.0579
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9893 - loss: 0.0385
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9916 - loss: 0.0342
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9931 - loss: 0.0289
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9967 - loss: 0.0149
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9952 - loss: 0.0212
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9947 - loss: 0.0195
Epoch 9/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9964 - loss: 0.0157
Epoch 10/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9976 - loss: 0.0103


In [ ]:
y_predicted = model.predict(X_test)

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, np.argmax(y_predicted, axis=1))
print(cm)

[[1949   70]
 [  50 1931]]


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, np.argmax(y_predicted, axis=1)))

              precision    recall  f1-score   support

           0       0.97      0.97      0.97      2019
           1       0.97      0.97      0.97      1981

    accuracy                           0.97      4000
   macro avg       0.97      0.97      0.97      4000
weighted avg       0.97      0.97      0.97      4000

